# Scenario 2: RAG Evaluation with Galileo

## What You'll Learn

In this notebook you will:

1. **Log RAG traces** that contain both a retriever span (the search step) and an LLM span (the generation step).
2. **Enable RAG-specific metrics** so Galileo automatically scores every trace for hallucination, relevance, and completeness.
3. **Create an evaluation dataset** — a reusable collection of test cases stored in Galileo.
4. **Run a prompt experiment** that sends every dataset row through a prompt template, calls the LLM, and scores the output.

---

## What is RAG?

**RAG** stands for **Retrieval-Augmented Generation**. Instead of asking an LLM to answer purely from its training data, you first *retrieve* relevant documents from a knowledge base, then pass those documents as context to the LLM so it can generate a grounded answer.

```
User Question
    │
    ▼
┌──────────────┐
│  Retriever   │  ← searches your knowledge base for relevant chunks
└──────┬───────┘
       │ chunks
       ▼
┌──────────────┐
│     LLM      │  ← generates an answer using the retrieved chunks as context
└──────┬───────┘
       │
       ▼
   Final Answer
```

---

## Key Concepts

| Concept | What It Means |
|---|---|
| **Retriever Span** | A span that logs the search/retrieval step — what query was sent and which document chunks came back. |
| **LLM Span** | A span that logs the generation step — what prompt was sent to the model and what it replied. |
| **Context Adherence** | *Does the answer stick to the retrieved context?* Catches hallucinations where the model invents facts not in the context. |
| **Chunk Relevance** | *Were the retrieved documents actually useful?* Scores each chunk against the original question. |
| **Dataset** | A collection of test cases (input/output pairs) stored in Galileo, reusable across experiments. |
| **Experiment** | Runs a prompt template over every row in a dataset, calls the LLM, and scores each output with the metrics you choose. |

---

## Prerequisites

- A `.env` file with `GALILEO_API_KEY` and `OPENAI_API_KEY` (see `.env.example`).
- Dependencies installed via `uv sync`.

### Load Environment Variables

Same pattern as Notebook 1 — we find the `.env` file in the repo root and load API keys (`GALILEO_API_KEY`, `OPENAI_API_KEY`) into the environment so the SDK and OpenAI client can authenticate.

In [7]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


### Imports and Setup

New imports compared to Notebook 1:

- **`GalileoLogger`** — used for *manual* trace logging. In the chatbot notebook, Galileo auto-instrumented OpenAI calls. Here we log retriever and LLM spans by hand so we can control exactly what gets recorded.
- **`galileo.datasets`** (`create_dataset`, `get_dataset`, `delete_dataset`) — for creating and managing evaluation datasets stored in Galileo.
- **`galileo.experiments`** (`run_experiment`, `PromptRunSettings`, `PromptTemplate`) — for running prompt experiments over datasets.

We also define a **`KNOWLEDGE_BASE`** dictionary that simulates a document store. In a real app this would be a vector database or search index; here it is a simple Python dict mapping questions to lists of text chunks.

In [ ]:
from galileo import GalileoLogger, GalileoMetrics, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.datasets import create_dataset, delete_dataset, get_dataset
from galileo.experiments import PromptRunSettings, PromptTemplate, run_experiment
from galileo.log_streams import create_log_stream, enable_metrics, get_log_stream
from galileo.projects import delete_project

PROJECT_NAME = 'GalileoEval_S2_RAG'
LOG_STREAM_NAME = 'rag-evaluation'
DATASET_NAME = 'rag-eval-dataset'
MODEL = 'gpt-4o-mini'
dataset_id = None

KNOWLEDGE_BASE = {
    'What is Galileo?': [
        'Galileo is an AI observability platform that helps teams evaluate, monitor, and improve LLM applications.',
        'Founded in 2021, Galileo provides metrics for RAG, agentic, and conversational AI systems.',
        "Galileo's Luna-2 model enables cost-effective evaluation at scale.",
    ],
    'How does RAG work?': [
        'RAG combines information retrieval with text generation.',
        'A retriever searches a knowledge base for relevant documents.',
        'The retrieved documents are passed as context to an LLM.',
    ],
    'What metrics does Galileo offer?': [
        'Galileo offers RAG metrics including context adherence, chunk relevance, and completeness.',
        'Agentic metrics include tool selection quality, action advancement, and agent efficiency.',
        'Safety metrics cover PII detection, toxicity, prompt injection, and bias detection.',
    ],
}

## 1. Initialize Galileo

Same pattern as Notebook 1: `galileo_context.init()` sets the active project and log stream. All traces logged after this call will appear under the project **GalileoEval_S2_RAG** in the Galileo console.

We also grab a `logger` instance and print the console URLs so you can jump straight to the UI to inspect traces.

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

log_stream = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
if not log_stream:
    log_stream = create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')

## 2. Log RAG Traces

Each trace represents one complete **question-to-answer flow**. A RAG trace contains **two spans**:

```
User Question
    │
    ├─► Retriever Span — logs the search step (input query → retrieved chunks)
    │
    └─► LLM Span — logs the generation step (prompt + context → model answer)
    │
    ▼
  Output (final answer returned to the user)
```

Here is what each method does:

| Method | Purpose |
|---|---|
| `logger.start_trace(input=...)` | Opens a new trace for one user question. |
| `logger.add_retriever_span(input=..., output=...)` | Records which chunks were retrieved and how long the search took (`duration_ns`). |
| `logger.add_llm_span(input=..., output=..., model=...)` | Records the prompt messages sent to the LLM, its response, token counts, and latency. |
| `logger.conclude(output=...)` | Closes the trace with the final answer. |
| `logger.flush()` | Sends all buffered traces to Galileo in one batch. |

We log three traces below. The **third trace intentionally has a WRONG answer** — it says "Galileo offers a free tier with unlimited API calls and supports over 200 programming languages", which is completely made up and not in the retrieved context. This lets us see how the **context adherence** metric catches hallucinations later.

In [10]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

chunks = KNOWLEDGE_BASE['What is Galileo?']
answer = 'Galileo is an AI observability platform founded in 2021 that helps teams evaluate and monitor LLM applications.'
logger.start_trace(input='What is Galileo?')
logger.add_retriever_span(input='What is Galileo?', output=chunks, duration_ns=50_000_000)
logger.add_llm_span(
    input=[
        Message(role=MessageRole.system, content='Answer from this context:\n' + '\n'.join(chunks)),
        Message(role=MessageRole.user, content='What is Galileo?'),
    ],
    output=Message(role=MessageRole.assistant, content=answer),
    model=MODEL,
    num_input_tokens=len(' '.join(chunks).split()),
    num_output_tokens=len(answer.split()),
    duration_ns=200_000_000,
)
logger.conclude(output=answer)

chunks = KNOWLEDGE_BASE['How does RAG work?']
answer = 'RAG combines retrieval and generation: a retriever finds relevant documents, then an LLM answers with that context.'
logger.start_trace(input='How does RAG work?')
logger.add_retriever_span(input='How does RAG work?', output=chunks, duration_ns=50_000_000)
logger.add_llm_span(
    input=[
        Message(role=MessageRole.system, content='Answer from this context:\n' + '\n'.join(chunks)),
        Message(role=MessageRole.user, content='How does RAG work?'),
    ],
    output=Message(role=MessageRole.assistant, content=answer),
    model=MODEL,
    num_input_tokens=len(' '.join(chunks).split()),
    num_output_tokens=len(answer.split()),
    duration_ns=200_000_000,
)
logger.conclude(output=answer)

chunks = KNOWLEDGE_BASE['What metrics does Galileo offer?']
answer = 'Galileo offers a free tier with unlimited API calls and supports over 200 programming languages.'
logger.start_trace(input='What metrics does Galileo offer?')
logger.add_retriever_span(input='What metrics does Galileo offer?', output=chunks, duration_ns=50_000_000)
logger.add_llm_span(
    input=[
        Message(role=MessageRole.system, content='Answer from this context:\n' + '\n'.join(chunks)),
        Message(role=MessageRole.user, content='What metrics does Galileo offer?'),
    ],
    output=Message(role=MessageRole.assistant, content=answer),
    model=MODEL,
    num_input_tokens=len(' '.join(chunks).split()),
    num_output_tokens=len(answer.split()),
    duration_ns=200_000_000,
)
logger.conclude(output=answer)

logger.flush()
'Logged 3 RAG traces'


'Logged 3 RAG traces'

## 3. Enable Core RAG Metrics

`enable_metrics()` tells Galileo which metrics to compute for every trace in this log stream. Here we enable the four foundational RAG metrics:

| Metric | What It Measures | Why It Matters |
|---|---|---|
| **`context_adherence`** | Does the LLM's answer stick to the retrieved context? | Catches **hallucinations** — cases where the model invents facts that are not in the provided chunks. The wrong answer in trace 3 should score low here. |
| **`chunk_relevance`** | Were the retrieved chunks actually relevant to the question? | Helps you debug your **retriever** — if chunks are irrelevant, the LLM has bad context to work with. |
| **`completeness`** | Does the answer cover all the important information from the context? | Detects answers that are technically correct but **leave out key details**. |
| **`context_relevance`** | Is the overall retrieved context relevant to the user's query? | A holistic score across all chunks (vs. chunk_relevance which scores each chunk individually). |

After running this cell, Galileo will automatically score new (and existing) traces in this log stream using these metrics.

In [11]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.context_adherence,
        GalileoMetrics.chunk_relevance,
        GalileoMetrics.completeness,
        GalileoMetrics.context_relevance,
    ],
)


[]

## 4. Enable Additional RAG Metrics

Now we add three more advanced metrics on top of the core four:

| Metric | What It Measures |
|---|---|
| **`context_precision`** | How much of the retrieved context is actually useful? A high score means most of the context was relevant (low noise). |
| **`precision_at_k`** | Precision of the top-K retrieved chunks — are the highest-ranked chunks the most relevant ones? |
| **`chunk_attribution_utilization`** | What fraction of the retrieved chunks actually contributed to the final answer? Low scores mean you are retrieving more than you need. |

### Important: Replace Behavior

Each `enable_metrics()` call **replaces** the entire metric set for the log stream. It does NOT add to the previous set. So when adding new metrics here, you must **re-include all metrics from Step 3** or they will be disabled.

### Important: Composite Metric Dependencies

`context_precision` and `precision_at_k` are *composite* metrics that depend on **Chunk Relevance** as a base metric. If you enable these composites without also enabling `chunk_relevance`, Galileo will return an error: *"Cannot disable 'Chunk Relevance' (required by: Precision@K, Context Precision)."* That is why we include `chunk_relevance` in the list below.

In [12]:
# Include all metrics from step 3 plus the new ones. Each enable_metrics() call replaces
# the log stream's metric set; omitting previous metrics would "disable" them and can
# trigger the Chunk Relevance error (composites require it to stay enabled).
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.context_adherence,
        GalileoMetrics.chunk_relevance,
        GalileoMetrics.completeness,
        GalileoMetrics.context_relevance,
        GalileoMetrics.context_precision,
        GalileoMetrics.precision_at_k,
        GalileoMetrics.chunk_attribution_utilization,
    ],
)


[]

## 5. Create an Evaluation Dataset

### What is a Dataset in Galileo?

A **dataset** is a collection of test cases stored in Galileo that you can reuse across multiple experiments. Each row has:

- **`input`** — the question or prompt (what gets sent to the model).
- **`output`** — the expected/reference answer (used by metrics like `ground_truth_adherence` to check if the model's response matches).

Datasets are persistent — once created, they live in your Galileo project and can be used by any experiment.

### What the Code Below Does

1. Checks if a dataset with this name already exists (`get_dataset`). If not, creates one with three initial rows.
2. Calls `ds.add_rows()` to **incrementally add** a fourth test case. This is useful when you want to expand your dataset over time without recreating it from scratch.

In [ ]:
ds = get_dataset(name=DATASET_NAME)
if not ds:
    ds = create_dataset(
        name=DATASET_NAME,
        content=[
            {"input": "What is Galileo?", "output": "Galileo is an AI observability platform for evaluating LLM applications."},
            {"input": "How does RAG work?", "output": "RAG combines retrieval of relevant documents with LLM generation."},
            {"input": "What metrics does Galileo offer?", "output": "Galileo offers RAG metrics, agentic metrics, and safety metrics."},
        ],
        project_name=PROJECT_NAME,
    )
dataset_id = ds.id
ds.add_rows([
    {
        "input": "What is context adherence?",
        "output": "Context adherence measures how well an LLM response aligns with the provided context.",
    },
])

## 6. Run a Prompt Experiment

### What is an Experiment?

An **experiment** takes a **prompt template**, runs it over **every row** in a dataset, calls the LLM for each row, and **scores each output** using the metrics you specify. The results are visible in the Galileo Experiments UI, where you can compare runs side by side.

### How It Works

1. **Prompt Template** — A list of messages with `{{input}}` as a placeholder. For each dataset row, `{{input}}` is replaced with that row's `input` value before calling the LLM.
2. **`model_alias`** — Must match a model name configured in your Galileo integrations (set up in the Galileo console under Integrations). If you see *"Model X is not available in any of your integrations"*, use the helper cell below to list available model aliases.
3. **Metrics** — Each metric is scored per-row. `ground_truth_adherence` compares the model's output to the dataset's reference `output` column, so it measures how close the LLM's answer is to your expected answer.

The first code cell below is optional — it lists the model aliases available in your Galileo integrations so you can pick the right one for `model_alias`.

## 8. Clean Up

Delete the evaluation dataset and the project to keep your Galileo workspace tidy. This is optional — skip this cell if you want to keep the data for further exploration in the console.

In [ ]:
from galileo import Message, MessageRole
from galileo.prompts import GlobalPromptTemplates

templates = GlobalPromptTemplates()
prompt_template = templates.get(name="rag-prompt-comparison-v1-template")
if not prompt_template:
    prompt_template = templates.create(
        name="rag-prompt-comparison-v1-template",
        template=[
            Message(role=MessageRole.system, content="You are a knowledgeable AI assistant. Answer accurately and concisely."),
            Message(role=MessageRole.user, content="{{input}}"),
        ],
        project_name=PROJECT_NAME,
    )

run_experiment(
    experiment_name="rag-prompt-comparison-v1",
    dataset=DATASET_NAME,
    prompt_template=prompt_template,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.ground_truth_adherence,
    ],
    prompt_settings=PromptRunSettings(model_alias="GPT-4o mini", max_tokens=150),
    project=PROJECT_NAME,
)
"Experiment complete"

## 8. Clean Up the Dataset and Project


In [ ]:
try:
    if dataset_id is not None:
        delete_dataset(id=str(dataset_id))
    else:
        delete_dataset(name=DATASET_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

delete_project(name=PROJECT_NAME)
